## Connettori a modelli (LLM)

In [3]:
import os
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI   # pip install langchain-openai

model = ChatOpenAI(
    openai_api_key=OPENAI_KEY, 
    model_name = "gpt-4o-mini",
    temperature=0.7, #livello di creatività/casualità nelle risposte (0: minima variabilità, 1: massima variabilità)
    max_tokens=1024, #numero massimo di token che il modello può generare in una sola risposta
    request_timeout=30,
    # ---
    # Possiamo usare questo connettore anche per modelli in locale
    # sfruttando applicativi che dialogano con il connettore similmente
    # alle API in cloud di OpenAI (ad es. LM-Studio)
    # ---
    #model="meta-llama/Llama-3.2-3B-Instruct", 
    #base_url = "http://127.0.0.1:8000/v1",
)

# Le info sui pricing dei modelli si possono trovare qui: 
# https://platform.openai.com/docs/pricing

### `init_chat_model`

La funzione `init_chat_model` consente di inizializzare un modello di chat (chat model) in modo unificato, agnostico rispetto al provider (ad esempio OpenAI, Anthropic, Google, Ollama …) in LangChain v1.0.
Prende come input una stringa che identifica il modello (es. `"gpt-4o"`, `"claude-3-opus-20240229"`), opzionalmente il provider, e altri eventuali parametri di configurazione.

**Caratteristiche principali**
- Ingresso semplificato: basta invocare `init_chat_model("nome-modello", model_provider="fornitore", …)` per ottienere un’istanza pronta all’uso.
- Supporto multiprovider: il provider può essere inferito automaticamente (ad es. prefissi “gpt-” → openai, “claude” → anthropic) oppure specificato manualmente.
- Compatibilità futura e commutabilità provider: grazie a questa astrazione, cambiare modello o provider richiede modifiche minime al codice applicativo, favorendo l’interoperabilità.


Con la release 1.0 di LangChain è stato scelto di snellire il namespace e focalizzarsi sui building-block fondamentali per applicazioni agentiche.
Tra questi, si è evinta la necessità di un helper “universale” per inizializzare modelli di chat. Inoltre, la crescente varietà e complessità dei modelli disponibili (diversi provider, modalità di invocazione, feature specifiche) richiedeva un’interfaccia che nascondesse queste complessità e offrisse un’esperienza omogenea.

In [5]:
from langchain.chat_models import init_chat_model

#  init_chat model

model = init_chat_model(
    "openai:gpt-4o-mini",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

In [4]:
model.invoke("Ciao, chi sei?")

AIMessage(content="Ciao! Sono un'intelligenza artificiale creata per assisterti con informazioni e rispondere alle tue domande. Come posso aiutarti oggi?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 13, 'total_tokens': 45, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb2CNeT6PogUK5fXPCTs9RneUZ9Kd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--93d17c94-a79a-4c21-9a18-668aab6b94d9-0', usage_metadata={'input_tokens': 13, 'output_tokens': 32, 'total_tokens': 45, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
model.invoke("""Agisci come un consulente di intelligenza artificiale esperto in trasformazione digitale per aziende manifatturiere.

Il tuo task è di elaborare una strategia operativa per introdurre soluzioni di AI e automazione nei processi produttivi.

Contesto: l’azienda cliente produce componenti meccanici di precisione, dispone di dati storici sulle linee di produzione ma non ha ancora digitalizzato completamente i flussi informativi.
Fornisci un piano strutturato che includa analisi dei dati, selezione delle tecnologie, roadmap di implementazione e metriche di impatto, spiegando le motivazioni dietro ogni scelta.""")

AIMessage(content="### Strategia Operativa per l'Introduzione di Soluzioni di AI e Automazione nella Produzione di Componenti Meccanici di Precisione\n\n#### Obiettivo\nL'obiettivo è ottimizzare i processi produttivi attraverso l'implementazione di tecnologie di intelligenza artificiale e automazione, migliorando l'efficienza, riducendo i costi e aumentando la qualità dei prodotti.\n\n---\n\n### 1. **Analisi dei Dati**\n\n**Motivazione:** I dati storici sulle linee di produzione rappresentano una risorsa fondamentale per l'implementazione delle soluzioni di AI. Un'analisi approfondita consente di identificare modelli, inefficienze e opportunità di miglioramento.\n\n**Fasi:**\n- **Raccolta Dati:** Raccogliere tutti i dati storici disponibili relativi alla produzione, ai guasti, ai tempi di fermo e alle variabili di processo.\n- **Pulizia e Normalizzazione:** Assicurarsi che i dati siano puliti, completi e normalizzati per un'analisi efficace.\n- **Analisi Descrittiva:** Utilizzare tecni

## Interfaccia Runnable

Per rendere più semplice la creazione di catene di eventi/esecuzione anche molto complesse i componenti di LangChain implementano tutti un protocollo "runnable" tramite un'interfaccia comune che permette di usare qualsiasi componente in modo standard; di seguito sono elencati i 3 principali metodi:

* **stream** - inviare risposte parziali mentre vengono generate
* **invoke** - eseguire la catena su un input
* **batch** - esecuzione della catena su più input

Uno dei vantaggi delle interfacce Runnable è dato dal fatto che dei componenti *runnable* possono essere concatenati in sequenze di esecuzione, facendo in modo che, automaticamente, gli output di un componente possano entrare in input ad un altro; il comando *pipe* `|` serve a questo e permette, nella sintassi LCEL (LangChain Expression Language) di creare componenti runnable partendo da altri componenti runnable, configurandoli in una sequenza di componenti che agiranno sinergicamente.

In [6]:
# streaming degli output

answer = None
for chunk in model.stream("elencami i primi 3 pianeti del sistema solare"):
    answer = chunk if answer is None else answer + chunk
    print(answer.text, "\n\n     -----\n\n")

 

     -----


I 

     -----


I primi 

     -----


I primi tre 

     -----


I primi tre pian 

     -----


I primi tre pianeti 

     -----


I primi tre pianeti del 

     -----


I primi tre pianeti del sistema 

     -----


I primi tre pianeti del sistema sol 

     -----


I primi tre pianeti del sistema solare 

     -----


I primi tre pianeti del sistema solare, 

     -----


I primi tre pianeti del sistema solare, part 

     -----


I primi tre pianeti del sistema solare, partendo 

     -----


I primi tre pianeti del sistema solare, partendo dal 

     -----


I primi tre pianeti del sistema solare, partendo dal Sole 

     -----


I primi tre pianeti del sistema solare, partendo dal Sole, 

     -----


I primi tre pianeti del sistema solare, partendo dal Sole, sono 

     -----


I primi tre pianeti del sistema solare, partendo dal Sole, sono:

 

     -----


I primi tre pianeti del sistema solare, partendo dal Sole, sono:

1 

     -----


I primi tre pianeti d

In [7]:
print(answer.content)

I primi tre pianeti del sistema solare, partendo dal Sole, sono:

1. **Mercurio** - È il pianeta più vicino al Sole e il più piccolo del sistema solare.
2. **Venere** - È il secondo pianeta dal Sole ed è simile alla Terra in termini di dimensioni, ma ha un'atmosfera densa e calda.
3. **Terra** - È il terzo pianeta dal Sole ed è l'unico noto per ospitare vita.

Se hai bisogno di ulteriori informazioni su ciascun pianeta, fammi sapere!


In [8]:
# gestione degli eventi durante lo streaming

async for event in model.astream_events("elencami i primi 3 pianeti del sistema solare"):

    if event["event"] == "on_chat_model_start":
        print(f"Input: {event['data']['input']}\n\n")

    elif event["event"] == "on_chat_model_stream":
        print(f"Token: {event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"\n\nFull message:\n{event['data']['output'].text}")

    else:
        pass

Input: elencami i primi 3 pianeti del sistema solare


Token: 
Token: I
Token:  primi
Token:  tre
Token:  pian
Token: eti
Token:  del
Token:  sistema
Token:  sol
Token: are
Token: ,
Token:  part
Token: endo
Token:  dal
Token:  Sole
Token: ,
Token:  sono
Token: :


Token: 1
Token: .
Token:  **
Token: Merc
Token: urio
Token: **

Token: 2
Token: .
Token:  **
Token: Ven
Token: ere
Token: **

Token: 3
Token: .
Token:  **
Token: Ter
Token: ra
Token: **


Token: Se
Token:  hai
Token:  bisogno
Token:  di
Token:  ulterior
Token: i
Token:  informazioni
Token:  su
Token:  cias
Token: cun
Token:  pian
Token: eta
Token: ,
Token:  fam
Token: m
Token: elo
Token:  sapere
Token: !
Token: 
Token: 
Token: 


Full message:
I primi tre pianeti del sistema solare, partendo dal Sole, sono:

1. **Mercurio**
2. **Venere**
3. **Terra**

Se hai bisogno di ulteriori informazioni su ciascun pianeta, fammelo sapere!


In [9]:
# esempio di chiamate in batch di più richieste
# da servire in parallelo (quando il sistema lo consente)
# con verifica del tempo di esecuzione

import time

queries = [
    "elencami i primi 3 pianeti del sistema solare",
    "elencami gli ultimi 3 pianeti del sistema solare",
    "elenca tre pianeti del sistema solare"
]

start_batch = time.time()
responses_batch = model.batch(queries)
end_batch = time.time()

print("=== Risultati batch ===")
for r in responses_batch:
    print(r.content[:80] + "...")
print(f"\nTempo totale batch: {end_batch - start_batch:.3f} secondi\n\n")

responses_single = []
start_single = time.time()
for q in queries:
    resp = model.invoke(q)
    responses_single.append(resp)
end_single = time.time()

print("=== Risultati singoli ===")
for r in responses_single:
    print(r.content[:80] + "...")
print(f"\nTempo totale chiamate singole: {end_single - start_single:.3f} secondi")

=== Risultati batch ===
I primi tre pianeti del sistema solare, in ordine di distanza dal Sole, sono:

1...
I tre pianeti più esterni del sistema solare, partendo da Saturno, sono:

1. **U...
Certo! Ecco tre pianeti del sistema solare:

1. Mercurio
2. Venere
3. Terra

Se ...

Tempo totale batch: 5.215 secondi


=== Risultati singoli ===
I primi tre pianeti del sistema solare, partendo dal Sole, sono:

1. **Mercurio*...
I tre ultimi pianeti del sistema solare, partendo da quelli più vicini al Sole, ...
Certo! Ecco tre pianeti del sistema solare:

1. Mercurio
2. Venere
3. Terra

Se ...

Tempo totale chiamate singole: 6.458 secondi


In [10]:
# restituzione di un output appena pronto,
# durante l'esecuzione di più input in batch

start_time = time.time()
for response in model.batch_as_completed([
    "Descrivi brevemente La Divina Commedia",
    "2 + 2 = ?",
    "Ciao, sono Enzo"
], config={
        'max_concurrency': 5,  # numero limite di chiamate in parallelo - inutile in questo esempio
    }):
    print(response[0], time.time() - start_time, "\n", response[1].content[:80] + "...", "\n\n")

2 0.8602638244628906 
 Ciao Enzo! Come posso aiutarti oggi?... 


1 1.0498533248901367 
 2 + 2 = 4.... 


0 5.576569318771362 
 "La Divina Commedia" è un poema epico scritto da Dante Alighieri nel XIV secolo.... 


